<a href="https://colab.research.google.com/github/langchain-samples/langsmith-studio-nb/blob/main/examples/deep_agent_in_studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://mybinder.org/v2/gh/langchain-samples/langsmith-studio-nb/HEAD?labpath=examples%2Fdeep_agent_in_studio.ipynb" target="_parent"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"/></a>
<a href="https://kaggle.com/kernels/welcome?src=https://github.com/langchain-samples/langsmith-studio-nb/blob/main/examples/deep_agent_in_studio.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# A Deep Agent in LangSmith Studio

Build a small Deep Agent in a notebook cell, then open [LangSmith Studio](https://docs.langchain.com/langsmith/studio)
on it and watch it call tools and write files.

Studio runs in your browser and talks to an agent server over HTTP. On Colab, Kaggle, and Binder
the kernel sits on a different machine than the browser, so that server needs a public URL.
`start_studio()` boots the server, opens a tunnel if the host needs one, and prints the link.

**You will need two API keys:**

| Key | Used for |
| --- | --- |
| `ANTHROPIC_API_KEY` | the agent's model calls |
| `LANGSMITH_API_KEY` | tracing, so each run shows up in LangSmith |

Get them from the [Anthropic Console](https://console.anthropic.com/settings/keys) and from
LangSmith under [Settings → API Keys](https://smith.langchain.com/settings/apikeys). A LangSmith
key starts with `lsv2_pt_`; it is shown exactly once, so copy it right away.

---

## 1. Store your keys

Where the keys live depends on where this notebook is running.

### Google Colab

Colab has a built-in secret store, and it is the only place you should put a key.

1. Click the 🔑 **key icon** in the left sidebar.
2. **Add new secret** → name it exactly `ANTHROPIC_API_KEY` → paste the value.
3. Turn on the **Notebook access** toggle. Without it the cell below cannot read the secret.
4. Repeat for `LANGSMITH_API_KEY`.

Secrets belong to your Google account rather than to this notebook, so you only do this once.

### Kaggle

Kaggle has a secret store too, under **Add-ons → Secrets**:

1. **Add-ons → Secrets → Add a new secret**.
2. Label it exactly `ANTHROPIC_API_KEY`, paste the value, **Save**.
3. Tick its **Attached** checkbox for this notebook. The kernel cannot see an unattached secret.
4. Repeat for `LANGSMITH_API_KEY`.

Also turn on **Internet** in the notebook settings. Without it the tunnel cannot start, and the
model call cannot leave the container either.

### Binder

**Binder has no secret store**, so the keys go in a `.env` file inside the session instead. Binder
is the one host here that clones the repo, so the template is already sitting next to you:

1. In the file browser on the left, open **`examples/`**, the folder this notebook is in, and find
   **`.env.example`**.
2. Right-click it → **Duplicate**.
3. Open the copy, replace the placeholders with your keys, and save with `Ctrl`/`Cmd`+`S`.
4. Rename it to exactly **`.env`**.

Fill it in before you rename it. By default a Jupyter server refuses to serve or create dotfiles at
all, and this repo lifts that restriction for Binder only, in `.binder/postBuild`, so that last
step can work.

That file lives in this session's container and dies with it. It does survive a kernel restart,
though, so you type the keys once rather than once per run.

> Never commit a `.env`. Binder builds from a *public* repository, so a committed key is readable
> by anyone, and Binder bakes it into an image other people can launch. `.env` is in this repo's
> `.gitignore` for exactly that reason.

### Local Jupyter or VS Code

Same `.env`, made from the same template:

```bash
cd examples
cp .env.example .env   # then fill in your keys
jupyter lab
```

Exported shell variables work too. The cell below reads whatever is already in the environment
before it looks anywhere else.

---

## 2. Install and load the secrets

`langsmith-studio-nb` pulls in `langgraph-cli[inmem]`, which is what serves the agent. Binder
builds this environment ahead of time from `.binder/`, so on Binder this cell has little left to do.

In [ ]:
%pip install -qq --progress-bar off \
  "deepagents~=0.7.8" \
  "langchain~=1.3.16" \
  "langchain-anthropic~=1.5.1" \
  "langsmith~=0.11.1" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

In [ ]:
import os

from dotenv import load_dotenv

from langsmith_studio_nb import load_secret

# Reads the .env sitting next to this notebook. Colab and Kaggle have no file to
# read, and load_secret asks their own secret stores instead.
load_dotenv()

load_secret("ANTHROPIC_API_KEY")
load_secret("LANGSMITH_API_KEY")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "studio-nb-example"

MODEL = "anthropic:claude-haiku-4-5"

print("Ready.")

`load_secret` checks the environment first, then asks whatever secret store the host it detects
happens to have. That means Colab's `userdata`, Kaggle's `UserSecretsClient`, and nothing at all on
the rest. So the two lines above cover every host, and `load_dotenv()` stacks in front of them,
because anything it puts in the environment wins.

It writes to `os.environ` for you, since that is where the model and tracing SDKs go looking.
Nothing prints a key. A notebook's output is the part that gets saved and shared, so a secret
should never land there. If a key is missing, the error names the fix for the host you are on.

Neither import needs installing. `python-dotenv` arrives with `langgraph-cli`, which
`langsmith-studio-nb` depends on.

---

## 3. Build the agent

`create_deep_agent` gives you a planner, a filesystem, and a subagent tool on top of whatever
tools you pass in. This one gets a single fake weather tool, so there is something to call and
something to write down without needing a third API key.

In [ ]:
from deepagents import create_deep_agent


def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"It's always sunny in {city}!"


agent = create_deep_agent(
    model=MODEL,
    tools=[get_weather],
    system_prompt=(
        "You are a travel assistant.\n"
        "Check the weather for every city the user mentions.\n"
        "Keep your working notes in files, and write your final answer to trip.md."
    ),
)

---

## 4. Let Studio trust the tunnel

**Colab, Kaggle, and Binder only.** Studio blocks agent servers on any host you have not told it
about, and all three reach your server through a Cloudflare tunnel. Allow the domain once:

1. Open [Studio](https://smith.langchain.com/studio/connect?mode=graph).
2. Click **Configure connection**.
3. Expand **Advanced Settings**.
4. Under **Allowed Domains**, add `*.trycloudflare.com`.
5. **Save**.

Add the wildcard, not the exact hostname Studio offers to add for you. That one changes every time
a tunnel opens. The list lives in your browser's local storage, so it is per browser and per user,
and there is no workspace-level setting. Skip all of this when you run locally, because there is no
tunnel to allow.

Without it, the link in the next cell looks like it does nothing at all.

---

## 5. Open Studio

`start_studio()` serves the notebook variable named `agent` and prints a link.

In [ ]:
from langsmith_studio_nb import start_studio

start_studio()

Open the link and give it something to do:

> I'm visiting Tokyo, Lisbon, and Oslo next month. What should I pack?

Studio calls the weather tool once per city, and shows the notes and `trip.md` appearing in the
agent's filesystem as it goes. A longer task would make it plan with a todo list first. A job this
small it just does. Every run lands in your LangSmith project as well.

Changed the agent? Re-run the cell that builds it, then this one. `start_studio()` stops the
previous server first. Hot reload is not available from a notebook, so that re-run is what picks up
your edit.

---

## Notes

- **The tunnel URL is public and unauthenticated** for as long as the cell runs, and anyone holding
  the link can drive your agent. Fine for a demo like this one. Think twice before pointing it at
  anything that touches real data or spends real money. The tunnel dies with the kernel.
- **Hosts reclaim idle runtimes.** Colab takes about 90 minutes, Binder about 10 with no browser
  activity. Either way you get a fresh tunnel URL when you reconnect. Binder also gives you a new
  container, so you have to make the `.env` again.
- **Several agents at once.** `start_studio("planner", "writer")` serves both variables over one
  tunnel and lets you switch between them in Studio's graph menu.
- **Running elsewhere?** `start_studio()` tunnels only when the browser cannot reach the kernel
  directly, which means Colab, Kaggle, Binder, and JupyterHub. Locally it serves `localhost` and
  skips the tunnel. Pass `tunnel=True` or `tunnel=False` when the guess is wrong.
- **`load_secret` works outside notebooks too.** It falls back to the plain environment, so the same
  cell runs unchanged in a script or in CI. Pass `required=False` for an optional key.

Full API and troubleshooting: the [README](https://github.com/langchain-samples/langsmith-studio-nb#readme).